# Exploration: What Predicts Default?

Seven business questions on the Home Credit data, each answered with SQL (DuckDB) and compared on default rate. Portfolio default rate: **8.07%** (307,511 applicants).

## Key findings

| # | Signal | Riskiest group | Default rate | Compared with | Default rate | Ratio | Share of applicants in riskiest group |
|---|---|---|---|---|---|---|---|
| 5 | External score (`EXT_SOURCE_2`) | Lowest 10% of scores | 18.35% | Highest 10% | 2.97% | **6.2x** | 10% |
| 3 | Home Credit payment history | Missed an instalment | 18.12% | Always on time | 6.79% | **2.7x** | 0.35% |
| 7 | Age | 20 to 29 | 11.44% | 60 to 69 | 4.92% | **2.3x** | 14.7% |
| 4 | Previous refusals | Refused 2+ times | 11.61% | Never refused | 7.07% | **1.6x** | 17.5% |
| 1 | Contract type | Cash loans | 8.35% | Revolving loans | 5.48% | **1.5x** | 90.5% |
| 2 | Bureau history | Has bad loans | 10.26% | No recorded problems | 7.00% | **1.5x** | 1.7% |
| 2 | Bureau history | Late, no bad loans | 9.51% | No recorded problems | 7.00% | **1.4x** | 22.8% |
| 3 | Home Credit payment history | Late, never missed | 9.35% | Always on time | 6.79% | **1.4x** | 50.1% |

## Takeaways

- **Every signal tested separates risk,** and most show a steady rise from safest to riskiest group (dose-response), which points to real relationships rather than noise.
- **Strength and reach differ.** A missed Home Credit payment is very strong but rare (0.35% of applicants); repeated refusals and lateness are weaker but cover far more applicants. A model needs both kinds.
- **A lender's own data beats external bureau data:** missed Home Credit payments (18.12%) are riskier than the worst bureau group (10.26%).
- **The external score is the strongest single signal** (6.2x spread) but its source is undisclosed, so it should be combined with interpretable features.
- **Splitting by degree matters:** "refused 2+ times" and "late" vs "missed" reveal risk that simple yes/no flags would blur.

## Open questions

- **Applicants new to Home Credit are the lowest-risk group (about 6%),** unlike applicants with no bureau record (10.12%). Bureau coverage and external scores explain only part of this (Q6).
- **Age** is likely a proxy for job stability and credit history length, and is a sensitive attribute in lending (Q7).

## Next step

These signals become features for the PD model: external scores, missed and late instalments (including how often and how late), number of previous refusals, bureau bad loans and lateness, and applicant characteristics. Each will be built at one row per applicant and joined to `application_train`.

In [30]:
import duckdb
import pandas as pd

APP    = 'data/raw/application_train.csv'
BUREAU = 'data/raw/bureau.csv'
INST   = 'data/raw/installments_payments.csv'
PREV   = 'data/raw/previous_application.csv'

# What does the portfolio look like, and where does the risk sit?

In [31]:
duckdb.sql(f'''
SELECT
    NAME_CONTRACT_TYPE,
    COUNT(*) AS applicants,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS portfolio_pct,
    ROUND(100.0 * AVG(TARGET), 2) AS default_rate,
    ROUND(AVG(AMT_CREDIT), 0) AS avg_credit
FROM '{APP}'
GROUP BY NAME_CONTRACT_TYPE
''').show()

┌────────────────────┬────────────┬───────────────┬──────────────┬────────────┐
│ NAME_CONTRACT_TYPE │ applicants │ portfolio_pct │ default_rate │ avg_credit │
│      varchar       │   int64    │    double     │    double    │   double   │
├────────────────────┼────────────┼───────────────┼──────────────┼────────────┤
│ Cash loans         │     278232 │         90.48 │         8.35 │   627966.0 │
│ Revolving loans    │      29279 │          9.52 │         5.48 │   324018.0 │
└────────────────────┴────────────┴───────────────┴──────────────┴────────────┘



In [32]:
duckdb.sql(f" select distinct(NAME_CONTRACT_TYPE) from '{APP}'").show()

┌────────────────────┐
│ NAME_CONTRACT_TYPE │
│      varchar       │
├────────────────────┤
│ Revolving loans    │
│ Cash loans         │
└────────────────────┘



### 1. Portfolio overview: where does the risk sit?

| Contract type | Applicants | Share of portfolio | Default rate | Avg loan amount |
|---|---|---|---|---|
| Cash loans | 278,232 | 90.48% | 8.35% | 627,966 |
| Revolving loans | 29,279 | 9.52% | 5.48% | 324,018 |

**Finding:** cash loans make up 90% of the portfolio, default about 1.5 times as often as revolving loans, and are about twice as large on average. Assuming the same loss rate for both, a cash loan carries roughly **3 times the expected loss** of a revolving loan (0.0835 × 627,966 ≈ 52,400 vs 0.0548 × 324,018 ≈ 17,800 units).

**Why it matters:** almost all of the portfolio's credit risk sits in cash loans, so that is where a PD model adds the most value.

*Notes: amounts are in an undisclosed currency (units). For revolving loans, `AMT_CREDIT` is the credit limit, not the amount drawn, so real exposure is lower. The current applications contain only cash and revolving loans; consumer (point-of-sale) loans appear only in customers' application history.*

# Does credit history at other lenders predict default?

In [33]:
duckdb.sql(f'''
    WITH bureau_features AS (
        SELECT
            SK_ID_CURR,
            COUNT(*) AS num_loans,
            SUM(CASE WHEN CREDIT_ACTIVE IN ('Sold', 'Bad debt') THEN 1 ELSE 0 END) AS num_bad_loans,
            MAX(CASE WHEN CREDIT_DAY_OVERDUE > 0 OR AMT_CREDIT_MAX_OVERDUE > 0 THEN 1 ELSE 0 END) AS ever_late
        FROM '{BUREAU}'
        GROUP BY SK_ID_CURR
    )
    SELECT
        CASE
            WHEN b.SK_ID_CURR IS NULL THEN '1. No bureau record'
            WHEN b.num_bad_loans > 0  THEN '2. Has bad loans'
            WHEN b.ever_late = 1      THEN '3. Late, no bad loans'
            ELSE                           '4. No recorded problems'
        END AS bureau_group,
        COUNT(*) AS applicants,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_applicants,
        ROUND(100.0 * AVG(a.TARGET), 2) AS default_rate
    FROM '{APP}' AS a
    LEFT JOIN bureau_features AS b
        ON a.SK_ID_CURR = b.SK_ID_CURR
    GROUP BY bureau_group
    ORDER BY bureau_group
''').show()

┌─────────────────────────┬────────────┬───────────────────┬──────────────┐
│      bureau_group       │ applicants │ pct_of_applicants │ default_rate │
│         varchar         │   int64    │      double       │    double    │
├─────────────────────────┼────────────┼───────────────────┼──────────────┤
│ 1. No bureau record     │      44020 │             14.31 │        10.12 │
│ 2. Has bad loans        │       5243 │               1.7 │        10.26 │
│ 3. Late, no bad loans   │      69988 │             22.76 │         9.51 │
│ 4. No recorded problems │     188260 │             61.22 │          7.0 │
└─────────────────────────┴────────────┴───────────────────┴──────────────┘



### 2. Does credit history at other lenders predict default?

A lender's best evidence of how someone will handle a loan is how they have handled loans before. The `bureau` table holds each applicant's track record at other lenders, so the question is whether a bad record elsewhere means a higher default rate with Home Credit.

Applicants are split into four groups:

| Group | Definition |
|---|---|
| 1. No bureau record | No loans at other lenders |
| 2. Has bad loans | At least one loan sold to collectors or written off |
| 3. Late, no bad loans | Overdue at some point (now or in the past), but no loan sold or written off |
| 4. No recorded problems | Loans at other lenders, never recorded as overdue |

**Method:** bureau is aggregated to one row per applicant (number of loans, number of bad loans, whether any loan was ever overdue), then LEFT JOINed to `application_train` so applicants without a bureau record are kept. Groups are assigned top to bottom, so an applicant with both bad loans and lateness counts as group 2.

*Note: a blank `AMT_CREDIT_MAX_OVERDUE` means the lender did not report it, not that the customer was never late. Group 4 is therefore "no recorded problems", not "proven clean".*

**Findings**

| Group | Applicants | Share | Default rate |
|---|---|---|---|
| 4. No recorded problems | 188,260 | 61.22% | 7.00% |
| 3. Late, no bad loans | 69,988 | 22.76% | 9.51% |
| 1. No bureau record | 44,020 | 14.31% | 10.12% |
| 2. Has bad loans | 5,243 | 1.70% | 10.26% |

- **Bureau history clearly separates risk.** Default rates climb step by step from 7.00% (no recorded problems) to 10.26% (bad loans).
- **Lateness matters even when every loan was eventually repaid.** Applicants who were overdue at some point default at 9.51%, about **1.4 times** the rate of applicants with no recorded problems. Splitting out lateness lowers the baseline from 7.68% (the earlier "clean" group) to 7.00%.
- **Volume tells a different story than rate.** Group 3 produces roughly 6,700 defaults, more than **12 times** as many as group 2 (about 540), because it is 13 times larger. Lateness is the more useful signal for the portfolio, even though bad loans are slightly riskier per applicant.
- **No credit history is almost as risky as a bad one** (10.12% vs 10.26%): with no track record, the lender cannot tell good payers from bad ones (thin-file risk).

**Why it matters:** number of bad loans, whether an applicant was ever late, and whether they have any bureau record at all are all worth building as model features.

## Does payment behaviour on previous Home Credit loans predict default?

In [34]:
duckdb.sql(f'''
    SELECT
        SK_ID_PREV,
        NUM_INSTALMENT_NUMBER,
        MIN(DAYS_INSTALMENT)                AS due_day,
        MAX(DAYS_ENTRY_PAYMENT)             AS last_payment_day,
        MAX(COALESCE(AMT_PAYMENT, 0))       AS largest_payment
    FROM '{INST}'
    WHERE SK_ID_PREV = 2050653
    GROUP BY SK_ID_PREV, NUM_INSTALMENT_NUMBER
    ORDER BY NUM_INSTALMENT_NUMBER
''').show()

┌────────────┬───────────────────────┬─────────┬──────────────────┬─────────────────┐
│ SK_ID_PREV │ NUM_INSTALMENT_NUMBER │ due_day │ last_payment_day │ largest_payment │
│   int64    │         int64         │ double  │      double      │     double      │
├────────────┼───────────────────────┼─────────┼──────────────────┼─────────────────┤
│    2050653 │                     1 │  -388.0 │           -396.0 │       51290.145 │
│    2050653 │                     2 │  -358.0 │           -361.0 │       51290.145 │
│    2050653 │                     3 │  -328.0 │           -323.0 │       48120.435 │
│    2050653 │                     4 │  -298.0 │           -309.0 │        180000.0 │
│    2050653 │                     5 │  -268.0 │           -267.0 │        43517.34 │
│    2050653 │                     6 │  -238.0 │           -240.0 │        43517.34 │
│    2050653 │                     7 │  -208.0 │           -218.0 │       729694.17 │
└────────────┴───────────────────────┴─────────┴──────

In [37]:
duckdb.sql('''SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_NUMBER,
            MIN(DAYS_INSTALMENT)          AS due_day,
            MAX(DAYS_ENTRY_PAYMENT)       AS last_payment_day,
            MAX(COALESCE(AMT_PAYMENT, 0)) AS largest_payment
        FROM 'data/raw/installments_payments.csv'
        WHERE SK_ID_CURR = 109083
        GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
        ''').show()

┌────────────┬────────────┬───────────────────────┬─────────┬──────────────────┬─────────────────┐
│ SK_ID_CURR │ SK_ID_PREV │ NUM_INSTALMENT_NUMBER │ due_day │ last_payment_day │ largest_payment │
│   int64    │   int64    │         int64         │ double  │      double      │     double      │
├────────────┼────────────┼───────────────────────┼─────────┼──────────────────┼─────────────────┤
│     109083 │    2050653 │                     2 │  -358.0 │           -361.0 │       51290.145 │
│     109083 │    1257553 │                     7 │  -591.0 │           -593.0 │       26583.525 │
│     109083 │    1257553 │                    12 │  -441.0 │           -444.0 │       26497.215 │
│     109083 │    1257553 │                     9 │  -531.0 │           -549.0 │       26583.525 │
│     109083 │    1257553 │                     5 │  -651.0 │           -689.0 │       26583.525 │
│     109083 │    1257553 │                     3 │  -711.0 │           -716.0 │       26583.525 │
│     1090

In [38]:
duckdb.sql(f'''
    WITH instalments AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_NUMBER,
            MIN(DAYS_INSTALMENT)          AS due_day,
            MAX(DAYS_ENTRY_PAYMENT)       AS last_payment_day,
            MAX(COALESCE(AMT_PAYMENT, 0)) AS largest_payment
        FROM '{INST}'
        WHERE SK_ID_CURR = 109083
        GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
    )
    SELECT
        SK_ID_CURR,
        COUNT(*) AS num_instalments,
        SUM(CASE WHEN largest_payment = 0 THEN 1 ELSE 0 END) AS num_missed,
        SUM(CASE WHEN last_payment_day - due_day > 0 AND largest_payment > 0 THEN 1 ELSE 0 END) AS num_late
    FROM instalments
    GROUP BY SK_ID_CURR
''').show()

┌────────────┬─────────────────┬────────────┬──────────┐
│ SK_ID_CURR │ num_instalments │ num_missed │ num_late │
│   int64    │      int64      │   int128   │  int128  │
├────────────┼─────────────────┼────────────┼──────────┤
│     109083 │              19 │          0 │        2 │
└────────────┴─────────────────┴────────────┴──────────┘



In [39]:
duckdb.sql(f'''
    WITH instalments AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_NUMBER,
            MIN(DAYS_INSTALMENT)          AS due_day,
            MAX(DAYS_ENTRY_PAYMENT)       AS last_payment_day,
            MAX(COALESCE(AMT_PAYMENT, 0)) AS largest_payment
        FROM '{INST}'
        GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
    ),
    payment_features AS (
        SELECT
            SK_ID_CURR,
            COUNT(*) AS num_instalments,
            SUM(CASE WHEN largest_payment = 0 THEN 1 ELSE 0 END) AS num_missed,
            SUM(CASE WHEN last_payment_day - due_day > 0 AND largest_payment > 0 THEN 1 ELSE 0 END) AS num_late
        FROM instalments
        GROUP BY SK_ID_CURR
    )
    SELECT
        CASE
            WHEN p.SK_ID_CURR IS NULL THEN '1. No HC payment history'
            WHEN p.num_missed > 0     THEN '2. Missed a payment'
            WHEN p.num_late > 0       THEN '3. Late, never missed'
            ELSE                           '4. Always on time'
        END AS payment_group,
        COUNT(*) AS applicants,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_applicants,
        ROUND(100.0 * AVG(a.TARGET), 2) AS default_rate
    FROM '{APP}' AS a
    LEFT JOIN payment_features AS p
        ON a.SK_ID_CURR = p.SK_ID_CURR
    GROUP BY payment_group
    ORDER BY payment_group
''').show()

┌──────────────────────────┬────────────┬───────────────────┬──────────────┐
│      payment_group       │ applicants │ pct_of_applicants │ default_rate │
│         varchar          │   int64    │      double       │    double    │
├──────────────────────────┼────────────┼───────────────────┼──────────────┤
│ 1. No HC payment history │      15868 │              5.16 │         5.98 │
│ 2. Missed a payment      │       1076 │              0.35 │        18.12 │
│ 3. Late, never missed    │     154142 │             50.13 │         9.35 │
│ 4. Always on time        │     136425 │             44.36 │         6.79 │
└──────────────────────────┴────────────┴───────────────────┴──────────────┘



**Findings**

| Group | Applicants | Share | Default rate |
|---|---|---|---|
| 1. No HC payment history | 15,868 | 5.16% | 5.98% |
| 4. Always on time | 136,425 | 44.36% | 6.79% |
| 3. Late, never missed | 154,142 | 50.13% | 9.35% |
| 2. Missed a payment | 1,076 | 0.35% | 18.12% |

- **A missed instalment is the strongest single signal found so far:** 18.12% default, about **2.7 times** the rate of customers who always paid on time, and well above the riskiest bureau group (10.26%). A lender's own repayment data is more predictive than external bureau data.
- **Paying late, even when every instalment was eventually paid, raises risk** from 6.79% to 9.35% (about 1.4 times).
- **"Ever late" is a broad flag:** half of all applicants paid at least one instalment late, often by only a few days. Measuring how often and how late (share of late instalments, average days late) should separate risk more sharply.
- **Applicants with no previous Home Credit instalments are the lowest-risk group (5.98%),** the opposite of the bureau result, where no record meant higher risk. The cause is not clear from this table; one possibility is customers new to Home Credit but established with other lenders. To be checked by combining payment and bureau history.

**Method:** payments were first aggregated to one row per instalment (last payment date, largest payment), so split payments and duplicated version rows are not double-counted, then to one row per customer. NULL and 0 payments were both treated as "nothing paid".

**Why it matters:** missed instalments, late instalments and the depth of lateness are strong candidate features for the PD model.

## Do previous refusals at Home Credit predict default?

In [40]:
duckdb.sql(f'''
    WITH refusal_features AS (
        SELECT
            SK_ID_CURR,
            COUNT(*) AS num_applications,
            SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END) AS num_refused
        FROM '{PREV}'
        GROUP BY SK_ID_CURR
    )
    SELECT
        CASE
            WHEN r.SK_ID_CURR IS NULL THEN '1. No previous applications'
            WHEN r.num_refused >= 2   THEN '2. Refused 2+ times'
            WHEN r.num_refused = 1    THEN '3. Refused once'
            ELSE                           '4. Never refused'
        END AS refusal_group,
        COUNT(*) AS applicants,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_applicants,
        ROUND(100.0 * AVG(a.TARGET), 2) AS default_rate
    FROM '{APP}' AS a
    LEFT JOIN refusal_features AS r
        ON a.SK_ID_CURR = r.SK_ID_CURR
    GROUP BY refusal_group
    ORDER BY refusal_group
''').show()

┌─────────────────────────────┬────────────┬───────────────────┬──────────────┐
│        refusal_group        │ applicants │ pct_of_applicants │ default_rate │
│           varchar           │   int64    │      double       │    double    │
├─────────────────────────────┼────────────┼───────────────────┼──────────────┤
│ 1. No previous applications │      16454 │              5.35 │         5.96 │
│ 2. Refused 2+ times         │      53760 │             17.48 │        11.61 │
│ 3. Refused once             │      46534 │             15.13 │         8.83 │
│ 4. Never refused            │     190763 │             62.03 │         7.07 │
└─────────────────────────────┴────────────┴───────────────────┴──────────────┘



**Findings**

| Group | Applicants | Share | Default rate |
|---|---|---|---|
| 1. No previous applications | 16,454 | 5.35% | 5.96% |
| 4. Never refused | 190,763 | 62.03% | 7.07% |
| 3. Refused once | 46,534 | 15.13% | 8.83% |
| 2. Refused 2+ times | 53,760 | 17.48% | 11.61% |

- **Default risk rises with every previous refusal:** 7.07% (never refused), 8.83% (once), 11.61% (two or more). A steady rise with the number of refusals (a dose-response pattern) suggests a genuine relationship rather than noise.
- **Splitting by count matters.** A simple "ever refused" flag would merge 8.83% and 11.61% and lose most of the signal.
- **Repeated refusal is both strong and common:** 1.6 times the never-refused rate, covering 17% of applicants (about 6,200 defaults). A missed payment (Q3) is riskier per applicant (18.12%) but applies to only 0.35% of applicants.
- **Applicants with no previous Home Credit applications are again the lowest-risk group (5.96%),** matching the payment-history result in Q3 (5.98%). The 586-applicant difference is people who applied before but never received a loan.

**Why it matters:** the number of previous refusals is a strong, widely available feature. Past refusals capture information about the applicant that the current application alone does not show.

## How well do the external credit scores separate risk?

In [41]:
duckdb.sql(f'''
    WITH scored AS (
        SELECT
            TARGET,
            EXT_SOURCE_2,
            NTILE(10) OVER (ORDER BY EXT_SOURCE_2) AS score_decile
        FROM '{APP}'
        WHERE EXT_SOURCE_2 IS NOT NULL
    )
    SELECT
        score_decile,
        COUNT(*) AS applicants,
        ROUND(MIN(EXT_SOURCE_2), 3) AS min_score,
        ROUND(MAX(EXT_SOURCE_2), 3) AS max_score,
        ROUND(100.0 * AVG(TARGET), 2) AS default_rate
    FROM scored
    GROUP BY score_decile
    ORDER BY score_decile
''').show()

┌──────────────┬────────────┬───────────┬───────────┬──────────────┐
│ score_decile │ applicants │ min_score │ max_score │ default_rate │
│    int64     │   int64    │  double   │  double   │    double    │
├──────────────┼────────────┼───────────┼───────────┼──────────────┤
│            1 │      30686 │       0.0 │     0.216 │        18.35 │
│            2 │      30685 │     0.216 │      0.34 │        12.08 │
│            3 │      30685 │      0.34 │      0.44 │         9.96 │
│            4 │      30685 │      0.44 │     0.512 │         8.36 │
│            5 │      30685 │     0.512 │     0.566 │         7.42 │
│            6 │      30685 │     0.566 │     0.608 │         6.65 │
│            7 │      30685 │     0.608 │     0.646 │         5.85 │
│            8 │      30685 │     0.646 │     0.682 │         4.89 │
│            9 │      30685 │     0.682 │     0.722 │          4.2 │
│           10 │      30685 │     0.722 │     0.855 │         2.97 │
└──────────────┴────────────┴─────

**Findings**

| Score decile | Score range | Default rate |
|---|---|---|
| 1 (lowest) | 0.000 to 0.216 | 18.35% |
| 2 | 0.216 to 0.340 | 12.08% |
| 3 | 0.340 to 0.440 | 9.96% |
| 4 | 0.440 to 0.512 | 8.36% |
| 5 | 0.512 to 0.566 | 7.42% |
| 6 | 0.566 to 0.608 | 6.65% |
| 7 | 0.608 to 0.646 | 5.85% |
| 8 | 0.646 to 0.682 | 4.89% |
| 9 | 0.682 to 0.722 | 4.20% |
| 10 (highest) | 0.722 to 0.855 | 2.97% |

Each decile holds about 30,685 applicants. 660 applicants (0.21%) have no `EXT_SOURCE_2` and were excluded.

- **A higher score means lower risk.** The default rate falls in every decile, from 18.35% to 2.97%: a **6 times** spread between the riskiest and safest 10%.
- **The relationship is perfectly monotonic,** which is the property a usable credit score needs.
- **The score separates best at the bottom:** the largest drops are between deciles 1, 2 and 3, so it is most useful for identifying the highest-risk applicants.
- **Compared with the signals built from history tables,** the lowest decile is as risky as applicants who missed a Home Credit payment (18.12%), but covers about 28 times as many applicants.
- Deciles hold equal numbers of applicants, not equal score ranges: most scores are concentrated between 0.5 and 0.7.

**Why it matters:** `EXT_SOURCE_2` is a strong, almost complete predictor and sets the benchmark for the model. Because its source is undisclosed, it can be measured but not explained, so the model should combine it with interpretable features (payment history, refusals, bureau history).

## Who are the applicants with no Home Credit history?

In [42]:
duckdb.sql(f'''
    WITH hc_customers AS (
        SELECT DISTINCT SK_ID_CURR
        FROM '{PREV}'
    ),
    bureau_customers AS (
        SELECT DISTINCT SK_ID_CURR
        FROM '{BUREAU}'
    )
    SELECT
        CASE WHEN h.SK_ID_CURR IS NULL THEN 'New to Home Credit'
             ELSE 'Returning customer' END AS customer_type,
        COUNT(*) AS applicants,
        ROUND(100.0 * AVG(CASE WHEN b.SK_ID_CURR IS NOT NULL THEN 1 ELSE 0 END), 2) AS pct_with_bureau_record,
        ROUND(AVG(a.EXT_SOURCE_2), 3) AS avg_ext_source_2,
        ROUND(100.0 * AVG(a.TARGET), 2) AS default_rate
    FROM '{APP}' AS a
    LEFT JOIN hc_customers     AS h ON a.SK_ID_CURR = h.SK_ID_CURR
    LEFT JOIN bureau_customers AS b ON a.SK_ID_CURR = b.SK_ID_CURR
    GROUP BY customer_type
''').show()

┌────────────────────┬────────────┬────────────────────────┬──────────────────┬──────────────┐
│   customer_type    │ applicants │ pct_with_bureau_record │ avg_ext_source_2 │ default_rate │
│      varchar       │   int64    │         double         │      double      │    double    │
├────────────────────┼────────────┼────────────────────────┼──────────────────┼──────────────┤
│ Returning customer │     291057 │                  85.72 │            0.513 │         8.19 │
│ New to Home Credit │      16454 │                  84.99 │            0.542 │         5.96 │
└────────────────────┴────────────┴────────────────────────┴──────────────────┴──────────────┘



**Findings**

| | Returning customer | New to Home Credit |
|---|---|---|
| Applicants | 291,057 | 16,454 |
| Have a bureau record | 85.72% | 84.99% |
| Average `EXT_SOURCE_2` | 0.513 | 0.542 |
| Default rate | 8.19% | 5.96% |

- **Theory tested:** new-to-Home-Credit applicants are lower risk because they are established with other lenders.
- **Not supported by bureau data:** both groups are equally likely to have a bureau record (85.0% vs 85.7%).
- **Partly supported by external scores:** new customers score slightly higher on average (0.542 vs 0.513), but both averages fall in the middle score deciles, so this explains only part of the gap in default rates.
- **The lower risk of new customers remains largely unexplained.** Possible reasons that this data cannot confirm: repeat borrowers may be customers who repeatedly need credit, or Home Credit may approve first-time applicants more cautiously.

**Why it matters:** "has previous Home Credit history" behaves differently from "has bureau history" and should enter the model as its own feature rather than being assumed to mean the same thing.

## Does the applicant’s age relate to default risk?

In [43]:
duckdb.sql(f'''
    SELECT
        FLOOR(-DAYS_BIRTH / 365.25 / 10) * 10 AS age_band,
        COUNT(*) AS applicants,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_applicants,
        ROUND(100.0 * AVG(TARGET), 2) AS default_rate
    FROM '{APP}'
    GROUP BY age_band
    ORDER BY age_band
''').show()

┌──────────┬────────────┬───────────────────┬──────────────┐
│ age_band │ applicants │ pct_of_applicants │ default_rate │
│  double  │   int64    │      double       │    double    │
├──────────┼────────────┼───────────────────┼──────────────┤
│     20.0 │      45186 │             14.69 │        11.44 │
│     30.0 │      82331 │             26.77 │         9.59 │
│     40.0 │      76599 │             24.91 │         7.64 │
│     50.0 │      68094 │             22.14 │         6.12 │
│     60.0 │      35301 │             11.48 │         4.92 │
└──────────┴────────────┴───────────────────┴──────────────┘



**Findings**

| Age band | Applicants | Share | Default rate |
|---|---|---|---|
| 20 to 29 | 45,186 | 14.69% | 11.44% |
| 30 to 39 | 82,331 | 26.77% | 9.59% |
| 40 to 49 | 76,599 | 24.91% | 7.64% |
| 50 to 59 | 68,094 | 22.14% | 6.12% |
| 60 to 69 | 35,301 | 11.48% | 4.92% |

- **Default risk falls steadily with age,** from 11.44% for applicants in their 20s to 4.92% for those in their 60s: about **2.3 times** lower.
- **Age needs no credit history,** so it remains informative for applicants with thin files, unlike the bureau, payment and refusal signals.
- **Correlation, not causation:** age likely reflects related factors such as job stability, income type (pensioners are concentrated in the oldest band) and length of credit history.
- **Age is a sensitive attribute.** Its use in lending decisions is restricted or requires justification in many jurisdictions; in practice this would need review before being used in a production model.

*Age computed as `-DAYS_BIRTH / 365.25`, grouped into 10-year bands.*